# Finding the Largest Active Fire Today (GEE + FIRMS)

## What You Will Learn
1. How to load the NASA FIRMS fire dataset in Google Earth Engine.
2. How to limit the search area with a GeoJSON AOI.
3. How to choose the latest available fire day, or a day before that.
4. How to group fire pixels into clusters and find the largest one.
5. How to export the results as GeoJSON files for later use.

## Datasets and Tools
1. Earth Engine provides the FIRMS dataset, which contains daily active fire detections.
2. `geemap` is used to display interactive maps inside the notebook.
3. Optional input: a GeoJSON AOI file or inline GeoJSON object to limit the analysis spatially.

## How To Use This Notebook
1. Edit the AOI setup code cell near the top if you want to focus on a smaller area.
2. Set `NUMBER_OF_DAYS_PRIOR` to choose the latest FIRMS day, or a day before that.
3. Run the notebook from top to bottom.
4. Read the print statements and comments as you go. They explain why each step exists.

## AOI and Date Controls

If you want to limit the fire search to one area or choose a different day, use the separate AOI setup code cell near the top of the notebook.

Set `AOI_INLINE_GEOJSON` first if you already have the geometry inside Python. If that is `None`, set `AOI_GEOJSON_PATH` to a local `.geojson` file.

Set `NUMBER_OF_DAYS_PRIOR = 0` for the latest available FIRMS day, `1` for the day before that, `2` for two days before that, and so on.


## Setup: install and access Earth Engine

This notebook uses FIRMS, geemap, and the Earth Engine API to identify the largest active fire in a study area. Make sure your Earth Engine project is active and your credentials are already authenticated before running the analysis cells.


In [17]:
import sys, geemap, ipywidgets, ipyleaflet
print(sys.executable)
print('geemap', getattr(geemap, 'version', 'missing'))
print('ipywidgets', getattr(ipywidgets, 'version', 'missing'))
print('ipyleaflet', getattr(ipyleaflet, 'version', 'missing'))



/Users/maples/miniconda3/envs/geo_env/bin/python
geemap missing
ipywidgets missing
ipyleaflet missing


In [18]:
# Install only the packages used in this Earth Engine workflow.
!pip install --upgrade earthengine-api geemap geopandas shapely


In [19]:
# Import Earth Engine so we can access datasets and run geospatial analysis.
import ee  # Earth Engine Python API

# Import geemap for interactive maps inside notebooks.
import geemap  # Map display helper for Earth Engine

project = 'sdm-gee-project-02'

# Reuse the same project everywhere instead of prompting for a new one.
# Try to initialize Earth Engine. If it fails, run authentication once.
try:
    ee.Initialize(project=project)  # Connect to Earth Engine using saved credentials
except Exception:
    ee.Authenticate()  # Open a browser-based login flow once
    ee.Initialize(project=project)  # Retry initialization with the same project

# Create a map so we can visualize results.
Map = geemap.Map()  # Interactive map widget

# Confirm that Earth Engine initialized successfully.
print('Earth Engine initialized.')

Earth Engine initialized.


## Notebook Setup

In [20]:
# Core imports for the Earth Engine onboarding workflow.
import json
from pathlib import Path

import ee
import geemap
import geopandas as gpd
from shapely.geometry import shape, mapping

# Keep the AOI logic simple and file-based for this workshop.


## Notebook Workflow

This notebook follows a simple pipeline, and each step prepares data for the next step:

1. **Load fire data**: We fetch the selected day's fire detections from NASA's FIRMS dataset via Google Earth Engine and optionally limit the search to a supplied GeoJSON AOI.
2. **Detect fire clusters**: We group nearby fire pixels together to identify distinct fire events.
3. **Find the largest fire**: We identify which fire cluster is the largest.
4. **Create a boundary polygon**: We convert the fire cluster into a clean polygon boundary.
5. **Export as GeoJSON**: We save the boundary as a standard geographic data format, plus a full-cluster check file and a top-10 summary file.

Each step builds on the previous one, creating a streamlined analysis pipeline that takes raw fire detection data and produces an actionable boundary polygon suitable for ordering high-resolution satellite imagery or emergency response planning.

Why this matters: beginners often need to see the full workflow before the details make sense. This notebook shows how a satellite fire dataset becomes a usable geographic boundary through filtering, clustering, polygon creation, and export.

The main strategy in this notebook is to start with a broad fire dataset, reduce it to one day, focus it with an AOI, then summarize the result into a single largest fire cluster.

## Earth Engine Authentication

Before you can use Google Earth Engine (GEE), you need to authenticate your account and initialize the API. There are several ways to do this, but this notebook uses the simplest browser-based path so you can get started quickly:

*   **Browser Authentication (Recommended for Beginners):** This is the easiest method. GEE will open a browser window to allow you to log in with your Google account.
*   **Service Account:** More suitable for automated scripts and server-side applications. Requires creating a service account in the Google Cloud Console and downloading credentials.

### Browser Authentication

The following code snippet initializes the Earth Engine API and authenticates your account using browser authentication. Run this cell first to authenticate.

**What happens when you run this cell:**

1. The code tries to initialize Earth Engine with `ee.Initialize()`
2. If you haven't authenticated yet, it will fail and catch the error
3. It then runs `ee.Authenticate()` which opens your web browser
4. You'll log in with your Google account and grant permissions
5. After successful authentication, it runs `ee.Initialize()` again to connect to Earth Engine
6. Finally, it prints a confirmation message

**Note:** You only need to authenticate once per computer. After that, you can just run `ee.Initialize()` in future sessions.

Why this step comes early: Earth Engine does the heavy data processing on Google's servers, but it still needs permission from your account before it will let you query datasets.

## Working Strategy

As you work through this notebook, keep these tips in mind:

- **Run cells in order**: Each cell depends on results from previous cells, so running them out of order may cause errors.
- **Make one change at a time**: When you edit the AOI or the day offset, rerun the AOI cell and then the analysis cells so you can see exactly what changed.
- **Watch the printed messages**: Print statements act like signposts. They tell you which AOI and which date the notebook is using.
- **Wait for Earth Engine**: Some steps take a few seconds because the work happens on Google's servers before the results come back to your notebook.
- **Check file paths**: When exporting files, make sure the `output/` directory exists. The code creates it automatically, but it helps to know where the results will land.
- **Expect a few passes**: Spatial workflows often require rerunning one or two cells while you adjust the AOI, the date, or the export settings. That is normal and part of the learning process.


## Core Concepts

Before we dive into the code, let's understand some core remote sensing and Google Earth Engine concepts that will help you understand what we're doing. These ideas explain why the code is written the way it is, not just what the code does:

### What is Remote Sensing?
Remote sensing is the science of obtaining information about objects or areas from a distance—typically using satellites. These satellites carry instruments (called "sensors") that measure different wavelengths of light reflected by Earth's surface. By analyzing these different wavelengths, we can identify fires, vegetation, water, and other features.

### What is Google Earth Engine (GEE)?
Google Earth Engine is a free cloud-based platform that holds decades of satellite imagery and makes it easy to analyze this data programmatically. Instead of downloading massive files to your computer, you send your analysis code to Google's servers, which do the heavy lifting and return just the results you need.

### Key GEE Objects You'll See

**ImageCollection**: A collection is like a "folder" of satellite images. The FIRMS dataset is an ImageCollection that contains fire detections from many satellite passes. Each image has multiple "bands" (different measurements of the same area—e.g., brightness temperature in different infrared wavelengths).

**Geometry**: In GEE, geometry defines the area of interest (AOI). It can be a point, line, polygon, or rectangle. In this notebook, you can supply a GeoJSON AOI file to focus the analysis on one place, or let it fall back to the whole world.

**Image Operations**: GEE lets you perform mathematical operations on images. For example, `fires.gt(100)` means "create a new image where pixels hotter than 100°C are marked as 1 (true) and colder pixels are marked as 0 (false)."

**Server-side vs. Client-side**: Most GEE operations happen on Google's servers (server-side), which is very fast. When you call `.getInfo()`, you're asking GEE to send the results back to your computer (client-side), which is slower. We try to minimize `.getInfo()` calls.

### What is a Reducer?
A reducer in GEE is a function that combines (reduces) data across space or time. For example, `ee.Reducer.minMax()` finds the minimum and maximum values across all pixels in an area. Think of it as "summarizing" a large image down to one or a few numbers.


In [21]:
AOI_INLINE_GEOJSON = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              112.41120682853011,
              -6.724298525297627
            ],
            [
              156.91347523128513,
              -8.19238030169457
            ],
            [
              166.79495973778148,
              -47.403239203760286
            ],
            [
              98.49433641071607,
              -45.913118817322
            ],
            [
              112.41120682853011,
              -6.724298525297627
            ]
          ]
        ]
      }
    }
  ]
}
AOI_GEOJSON_PATH = Path('notebooks/aussie.geojson')
NUMBER_OF_DAYS_PRIOR = 1


def _load_geojson_object(candidate):
    """Load GeoJSON from a file path, JSON string, or dictionary."""
    if isinstance(candidate, Path):
        if not candidate.exists():
            raise FileNotFoundError(f'AOI file not found: {candidate}')
        with candidate.open('r', encoding='utf-8') as fp:
            return json.load(fp), str(candidate)

    if isinstance(candidate, str):
        text_candidate = candidate.strip()
        possible_path = Path(candidate)
        if possible_path.exists():
            with possible_path.open('r', encoding='utf-8') as fp:
                return json.load(fp), str(possible_path)
        return json.loads(text_candidate), 'inline GeoJSON string'

    if isinstance(candidate, dict):
        return candidate, 'inline GeoJSON object'

    raise TypeError('AOI input must be a GeoJSON object, JSON string, or file path.')


def _geojson_to_ee_geometry(geojson_data):
    """Convert GeoJSON content into one Earth Engine geometry."""
    geojson_type = geojson_data.get('type')

    if geojson_type == 'FeatureCollection':
        features = geojson_data.get('features', [])
        if not features:
            raise ValueError('The AOI FeatureCollection does not contain any features.')
        geometries = [shape(feature['geometry']) for feature in features if feature.get('geometry')]
        if not geometries:
            raise ValueError('The AOI FeatureCollection does not contain any geometries.')
        return ee.Geometry(mapping(gpd.GeoSeries(geometries).unary_union))

    if geojson_type == 'Feature':
        if not geojson_data.get('geometry'):
            raise ValueError('The AOI Feature does not contain a geometry.')
        return ee.Geometry(geojson_data['geometry'])

    if geojson_type in {'Point', 'MultiPoint', 'LineString', 'MultiLineString', 'Polygon', 'MultiPolygon'}:
        return ee.Geometry(geojson_data)

    raise ValueError('AOI GeoJSON must be a FeatureCollection, Feature, or geometry object.')


def load_processing_geometry(aoi_inline_geojson=None, aoi_path=None):
    """Return an Earth Engine geometry and a label for the source AOI."""
    candidates = []

    if aoi_inline_geojson is not None:
        candidates.append(('inline GeoJSON', aoi_inline_geojson))

    if aoi_path is not None:
        candidates.append((f'path: {aoi_path}', aoi_path))

    if not candidates:
        raise ValueError('No AOI was provided. Set AOI_INLINE_GEOJSON or AOI_GEOJSON_PATH.')

    for label, candidate in candidates:
        try:
            geojson_data, _ = _load_geojson_object(candidate)
            return _geojson_to_ee_geometry(geojson_data), label
        except Exception:
            continue

    raise ValueError('Could not load a valid AOI GeoJSON object.')


geometry, aoi_source = load_processing_geometry(AOI_INLINE_GEOJSON, AOI_GEOJSON_PATH)
print(f'Using AOI from: {aoi_source}')


Using AOI from: inline GeoJSON


In [22]:
# ============================================================================
# STEP 1: Load the selected day's active fires from NASA's FIRMS dataset
# ============================================================================

# Load the FIRMS (Fire Information for Resource Management System) ImageCollection.
# This dataset contains near-real-time fire detections from satellite sensors.
dataset = ee.ImageCollection("FIRMS")

# Use the latest available FIRMS image as the reference point.
# This is safer than using the computer's calendar date, because the data archive may lag behind today.
latest_available_image = dataset.sort('system:time_start', False).first()
latest_available_date = ee.Date(latest_available_image.get('system:time_start'))

# Move backward from the latest available FIRMS date by the amount the user chose above.
target_fire_date = latest_available_date.advance(-NUMBER_OF_DAYS_PRIOR, 'day')
TARGET_FIRE_START = target_fire_date.format('yyyy-MM-dd').getInfo()
TARGET_FIRE_END = target_fire_date.advance(1, 'day').format('yyyy-MM-dd').getInfo()

# Keep only the images that fall within the selected day.
dataset_for_day = dataset.filterDate(TARGET_FIRE_START, TARGET_FIRE_END)
print(f'Using FIRMS date: {TARGET_FIRE_START} (NUMBER_OF_DAYS_PRIOR={NUMBER_OF_DAYS_PRIOR})')

# Get the most recent image in the collection
# sort(..., False) means sort in descending order (newest first)
# .first() gets just the first (most recent) image
lastimg = dataset_for_day.sort('system:time_start', False).first()

# Select the T21 band (brightness temperature in the thermal infrared)
# and create a binary image: pixels hotter than 100°C = 1, cooler pixels = 0
# The gt() function means "greater than"
fires = lastimg.select('T21').gt(100)

# Define how we want to visualize the fire temperature data on the map
# "min" and "max" set the scale—temperatures between 100°C and 500°C will be shown
# "palette" defines the color ramp: cooler fires (100°C) appear yellow,
# medium fires appear orange, and hottest fires appear red
firesVis = {
    "min": 100.0,      # Minimum temperature to display (in °C)
    "max": 500.0,      # Maximum temperature to display (in °C)
    "palette": ["yellow", "orange", "red"]  # Color ramp: cool to hot
}

# Create an interactive geemap Map object centered on the world
m = geemap.Map(center=[0, 0], zoom=2)

# Add the fire temperature layer to the map
# addLayer takes three arguments: the data to display, how to visualize it, and a label
m.addLayer(lastimg.select('T21'), firesVis, 'FIRMS T21 (Fire Temperature)')

# Display the map in the notebook
m

Using FIRMS date: 2026-09-21 (NUMBER_OF_DAYS_PRIOR=1)


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## Group Fire Pixels Into Clusters

In remote sensing, fires are often detected as individual pixels (grid cells) on a satellite image. When many pixels cluster together, they likely represent a single large fire event. Our next step is to group connected fire pixels into clusters.

Why this matters: a single satellite image can contain many fire pixels, but not every pixel is a separate fire. Clustering helps us turn lots of dots into one meaningful fire outline.

The `connectedPixelCount()` function does this by:
1. Looking at each fire pixel
2. Following "connected" paths to neighboring fire pixels (up, down, left, right, and diagonals)
3. Counting how many pixels form each connected group

This helps us identify that, for example, 47 pixels belong to fire cluster A, 12 pixels belong to fire cluster B, and so on. We'll then visualize these clusters and find the largest one.

For beginners, think of this as the difference between counting individual sparks and measuring the whole fire area.

In [23]:
# ============================================================================
# STEP 2: Identify fire clusters and find the largest one
# ============================================================================

# connectedPixelCount() counts how many fire pixels are connected to each pixel
# Arguments:
#   - 1000: maximum size to count (any cluster larger than 1000 pixels gets capped at 1000)
#   - True: use 8-directional connectivity (includes diagonals, not just up/down/left/right)
# The result is a new image where each pixel's value = size of its cluster
connectedCount = fires.connectedPixelCount(1000, True)

# Define visualization colors for the cluster sizes
# Small clusters will appear purple, large clusters will appear yellow
conn_vis = {
    'min': 0,          # Smallest possible cluster (0 pixels)
    'max': 500,        # Largest cluster size we expect to display (500 pixels)
    'palette': ['#4B0082', '#6A5ACD', '#8A2BE2', '#DA70D6', '#FFD700']  # Purple→indigo→yellow
}

# Use a reducer to find the minimum and maximum cluster sizes
# A reducer is a function that summarizes many values into one or a few values
# ee.Reducer.minMax() finds both the smallest and largest values in an area
minMax = connectedCount.reduceRegion(
    reducer=ee.Reducer.minMax(),  # Find min and max values
    geometry=geometry,             # Search across the entire world
    scale=1000,                    # Use 1 km resolution pixels for the calculation
    maxPixels=1e9                  # Allow processing up to 1 billion pixels (for performance)
)

# Print the results to see the range of cluster sizes
# Note: .getInfo() brings server-side GEE data to your computer (client-side)
print("Min and Max connected pixel counts:", minMax.getInfo())

# Add the cluster layer to our existing map
# This shows all fire clusters colored by size
m.addLayer(connectedCount, conn_vis, 'Fire Clusters by Size (purple→yellow)')

# Display the updated map
m

Min and Max connected pixel counts: {'T21_max': 284, 'T21_min': 3}


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [24]:
# ============================================================================
# STEP 3: Isolate the largest fire cluster
# ============================================================================

# Get the maximum cluster size from our previous calculation
# ee.Number() wraps the value so we can work with it in GEE
T21_max = ee.Number(minMax.get('T21_max'))

# Create a binary image: pixels belonging to the largest cluster = 1, all others = 0
# .eq() means "equals"—so this image marks only pixels equal to T21_max
largest_cluster = connectedCount.eq(T21_max)

# Apply a mask to hide all non-largest-cluster pixels
# .updateMask() makes pixels with value 0 transparent on the map
# Only the largest cluster (value 1) remains visible
largest_mask = largest_cluster.updateMask(largest_cluster)

# Add the largest fire cluster to the map in red for clear visibility
m.addLayer(largest_mask, {'min': 0, 'max': 1, 'palette': ['red']}, 'Largest Fire Cluster')

# Print the size of the largest cluster to the console
print('Size of largest cluster (T21_max):', T21_max.getInfo(), 'connected pixels')
print("Full statistics:", minMax.getInfo())

Size of largest cluster (T21_max): 284 connected pixels
Full statistics: {'T21_max': 284, 'T21_min': 3}


In [25]:
# ============================================================================
# STEP 4: Convert fire cluster pixels into polygon boundaries
# ============================================================================

# Convert the binary fire image to vectors (polygon geometries)
# This turns each connected fire patch into a clean polygon boundary.
# Arguments:
#   - reducer: ee.Reducer.countEvery() counts pixels (used for vector creation)
#   - geometry: search area (entire world)
#   - scale: 1 km resolution pixels
#   - maxPixels: max pixels to process
#   - eightConnected: use 8-directional connectivity (includes diagonals)
#   - geometryType='bb': create bounding boxes (rectangular polygons)
all_vectors = fires.reduceToVectors(
    reducer=ee.Reducer.countEvery(),
    geometry=geometry,
    scale=1000,
    maxPixels=1e9,
    eightConnected=True,
    geometryType='bb'
)

# For each polygon, calculate its area (in square meters)
# This function will be applied to each feature in the collection
def set_area(feat):
    """Calculate the area of a feature's geometry."""
    geom = feat.geometry()
    # .area(maxError) computes area with specified precision (maxError=1 meter)
    # Server-side operations use non-zero maxError for efficiency
    area_m = geom.area(1)
    # .set() adds a new property to the feature
    return feat.set('area', area_m)

# Apply the set_area function to all polygons
# .map() applies a function to each item in a collection
vectors = all_vectors.map(set_area)

# Keep the ten largest polygons so we can export a small folder of fire boundaries.
# sort(..., False) sorts by area in descending order (largest first)
# .limit(10) keeps only the first ten features after sorting
top_10_features = ee.FeatureCollection(vectors.sort('area', False).limit(10))

# The biggest fire is still the first feature in the sorted top-10 collection.
largest_feature = ee.Feature(top_10_features.first())

# Calculate the area in square kilometers for readability
# .getInfo() brings the server-side calculation result to your computer
area_m = largest_feature.geometry().area(1)
area_m_val = area_m.getInfo()  # Fetch from GEE server to Python
area_km = area_m_val / 1e6     # Convert square meters to square kilometers

print(f'Largest fire area: {area_m_val:,.0f} m² ({area_km:,.1f} km²)')

# Add the polygon boundaries to the map
m.addLayer(vectors, {'color': 'orange'}, 'All cluster polygons')
m.addLayer(ee.FeatureCollection(largest_feature), {'color': 'red', 'fillColor': '00000000'}, 'Largest fire (boundary)')

# Center the map on the largest fire and zoom in
m.centerObject(largest_feature, 10)

# Display the updated map
m

Largest fire area: 1,442,644,842 m² (1,442.6 km²)


Map(bottom=812.0, center=[-18.984193823019872, 138.5483605474495], controls=(WidgetControl(options=['position'…

In [26]:
# ============================================================================
# STEP 5: Convert to GeoJSON format (for sharing and interoperability)
# ============================================================================

import json  # Python's built-in module for working with JSON data

# Convert the GEE Feature to a Python dictionary (GeoJSON format)
# GeoJSON is a standard format for geographic data that works across many tools
# .getInfo() brings the server-side feature data to the client (your computer)
largest_feature_info = largest_feature.getInfo()

# Get the full top-10 collection as a regular Python dictionary.
# We use this later to write one GeoJSON file per fire cluster.
top_10_features_info = top_10_features.getInfo()

# Wrap the single feature in a FeatureCollection
# (GeoJSON standard: a collection of features)
feature_collection_geojson = {
    "type": "FeatureCollection",
    "features": [largest_feature_info]
}

# Display the GeoJSON structure in a readable format
# This shows you what the largest fire looks like before saving it
print("GeoJSON Output for the largest fire:")
print(json.dumps(feature_collection_geojson, indent=2))

# Also show a short summary of the top 10 clusters.
print("\nTop 10 fire clusters:")
for idx, feature in enumerate(top_10_features_info['features'], start=1):
    area_m = feature['properties'].get('area', 0)
    area_km = area_m / 1e6
    print(f"{idx:02d}. area = {area_m:,.0f} m² ({area_km:,.1f} km²)")

GeoJSON Output for the largest fire:
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "geodesic": false,
        "type": "Polygon",
        "coordinates": [
          [
            [
              138.29489910961595,
              -19.10587017404143
            ],
            [
              138.8018219852845,
              -19.10587017404143
            ],
            [
              138.8018219852845,
              -18.862461788135924
            ],
            [
              138.29489910961595,
              -18.862461788135924
            ],
            [
              138.29489910961595,
              -19.10587017404143
            ]
          ]
        ]
      },
      "id": "+34575+12131",
      "properties": {
        "area": 1442644841.8477206,
        "count": 284,
        "label": 1
      }
    }
  ]
}

Top 10 fire clusters:
01. area = 1,442,644,842 m² (1,442.6 km²)
02. area = 290,880,596 m² (290.9 km²)
03. area = 2

## Export Results

Now we'll save the fire boundary polygons to GeoJSON files.
We will keep the single largest-fire GeoJSON and also create a folder with one GeoJSON file for each of the ten largest fire clusters.

Why this matters: exporting gives you a portable result. A GeoJSON file can be opened in GIS software, used as an AOI for another notebook, or shared with someone who is not working in Earth Engine.

These files can be:
- Imported into mapping software like QGIS, ArcGIS, or Google Earth
- Used as Areas of Interest (AOIs) for follow-up analysis
- Shared with other researchers or emergency responders
- Used in web mapping applications

When you read the export cell, pay attention to the filenames. The notebook uses the selected date in each output name so you can keep track of which run produced which file.


In [27]:
# ============================================================================
# Save the GeoJSON to a file
# ============================================================================

from pathlib import Path  # Python's cross-platform filesystem path handling

# Create the output directory if it doesn't exist
# parents=True: create parent directories if needed
# exist_ok=True: don't error if directory already exists
output_dir = Path("output")
# Get the date from the image's system:time_start property
# This is the acquisition date of the fire detection data
image_date = ee.Date(lastimg.get('system:time_start'))

# Format the date as yyyy-MM-dd for the filename.
# In Earth Engine, lower-case yyyy is the calendar year and lower-case dd is the day of month.
# Using lowercase year and day tokens keeps the file name readable and correct.
date_string = image_date.format('yyyy-MM-dd').getInfo()

# Create the output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

# Specify the GeoJSON file path for the largest fire.
# This keeps the original file name so downstream notebooks still work.
geojson_path = output_dir / f"{date_string}_largest_fire_cluster.geojson"

# Write the GeoJSON to a file
# json.dump() serializes (converts) the Python dictionary to JSON format
# ensure_ascii=False: preserves special characters (like accents)
# indent=2: makes the file human-readable with 2-space indentation
with geojson_path.open("w", encoding="utf-8") as fp:
    json.dump(feature_collection_geojson, fp, ensure_ascii=False, indent=2)
# Create a combined GeoJSON file that contains every fire cluster.
# This is a simple sanity check so we can inspect all of the polygons together.
all_clusters_geojson_path = output_dir / f"{date_string}_all_fire_clusters.geojson"
all_clusters_geojson = {
    "type": "FeatureCollection",
    "features": all_vectors.getInfo()['features']
}
with all_clusters_geojson_path.open("w", encoding="utf-8") as fp:
    json.dump(all_clusters_geojson, fp, ensure_ascii=False, indent=2)

# Create a folder for the top 10 fire clusters.
# Each fire cluster gets its own GeoJSON file so the outputs are easy to inspect.
top_10_dir = output_dir / f"{date_string}_top10_largest_fire_clusters"
top_10_dir.mkdir(parents=True, exist_ok=True)

# Remove old GeoJSON files from earlier runs so the folder always reflects the newest export.
for old_geojson in top_10_dir.glob('*.geojson'):
    old_geojson.unlink()

# Keep a ranked list so we can also write one combined GeoJSON file for the top 10 fires.
top_10_ranked_features = []

# Write one GeoJSON file per fire cluster.
# We keep the features in sorted order so file 01 is the largest fire, file 02 is the next largest, and so on.
for rank, feature in enumerate(top_10_features_info['features'], start=1):
    # Make a copy of the feature so we can add a simple rank label for the exported file.
    ranked_feature = {
        "type": "Feature",
        "geometry": feature["geometry"],
        "properties": dict(feature.get("properties", {})),
    }
    ranked_feature["properties"]["rank"] = rank
    ranked_feature["properties"]["date"] = date_string

    ranked_geojson = {
        "type": "FeatureCollection",
        "features": [ranked_feature]
    }

    top_10_ranked_features.append(ranked_feature)

    top_10_path = top_10_dir / f"{date_string}_largest_fire_cluster_{rank:02d}.geojson"
    with top_10_path.open("w", encoding="utf-8") as fp:
        json.dump(ranked_geojson, fp, ensure_ascii=False, indent=2)
# Write one combined GeoJSON file containing the 10 largest fires.
top_10_collection_geojson_path = output_dir / f"{date_string}_top10_largest_fire_clusters.geojson"
top_10_collection_geojson = {
    "type": "FeatureCollection",
    "features": top_10_ranked_features
}
with top_10_collection_geojson_path.open("w", encoding="utf-8") as fp:
    json.dump(top_10_collection_geojson, fp, ensure_ascii=False, indent=2)

# Print confirmation messages showing where the files were saved
print(f"✓ Largest fire GeoJSON exported to: {geojson_path.resolve()}")
print(f"✓ All fire clusters GeoJSON exported to: {all_clusters_geojson_path.resolve()}")
print(f"✓ Top 10 fire clusters GeoJSON exported to: {top_10_collection_geojson_path.resolve()}")
print(f"✓ Top 10 fire cluster GeoJSON files exported to: {top_10_dir.resolve()}")

✓ Largest fire GeoJSON exported to: /Users/maples/Github/SGC-Quickstart/notebooks/output/2026-09-21_largest_fire_cluster.geojson
✓ All fire clusters GeoJSON exported to: /Users/maples/Github/SGC-Quickstart/notebooks/output/2026-09-21_all_fire_clusters.geojson
✓ Top 10 fire clusters GeoJSON exported to: /Users/maples/Github/SGC-Quickstart/notebooks/output/2026-09-21_top10_largest_fire_clusters.geojson
✓ Top 10 fire cluster GeoJSON files exported to: /Users/maples/Github/SGC-Quickstart/notebooks/output/2026-09-21_top10_largest_fire_clusters
